# Векторизация четырёх текстов

Метод: TF-IDF-векторизация слов и биграмм, затем косинусная близость между векторами текстов.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts_dir = Path("texts")
files = sorted(texts_dir.glob("*.txt"))

texts = [path.read_text(encoding="utf-8") for path in files]
names = [path.stem for path in files]

pd.DataFrame({
    "file": [path.name for path in files],
    "length_chars": [len(text) for text in texts]
})

,file,length_chars
0,text1_corpus_linguistics.txt,2807
1,text2_language_change.txt,2724
2,text3_urban_transport.txt,2637
3,text4_literary_rain.txt,2677


In [2]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[а-яё]{3,}\b",
    ngram_range=(1, 2),
    min_df=1
)

X = vectorizer.fit_transform(texts)
similarity = cosine_similarity(X)

sim_df = pd.DataFrame(similarity, index=names, columns=names)
sim_df.round(3)

,text1_corpus_linguistics,text2_language_change,text3_urban_transport,text4_literary_rain
text1_corpus_linguistics,1.000,0.117,0.051,0.028
text2_language_change,0.117,1.000,0.058,0.051
text3_urban_transport,0.051,0.058,1.000,0.036
text4_literary_rain,0.028,0.051,0.036,1.000


In [3]:
pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append({
            "text_1": names[i],
            "text_2": names[j],
            "cosine_similarity": similarity[i, j]
        })

pairs_df = pd.DataFrame(pairs).sort_values("cosine_similarity", ascending=False)
pairs_df.round(3)

,text_1,text_2,cosine_similarity
0,text1_corpus_linguistics,text2_language_change,0.117
3,text2_language_change,text3_urban_transport,0.058
4,text2_language_change,text4_literary_rain,0.051
1,text1_corpus_linguistics,text3_urban_transport,0.051
5,text3_urban_transport,text4_literary_rain,0.036
2,text1_corpus_linguistics,text4_literary_rain,0.028


## Комментарий к результату

Самыми близкими оказались тексты `text1_corpus_linguistics` и `text2_language_change`. Я ожидал этого результата, потому что оба текста относятся к лингвистике и написаны в похожем учебно-научном стиле. В них часто встречаются слова, связанные с языком, текстами, корпусом, конструкциями, частотностью, исследованием и употреблением, поэтому TF-IDF-векторы этих текстов получились наиболее похожими.